In [7]:
import pandas as pd  # Pandas for data manipulation and handling DataFrame objects
from sklearn.model_selection import train_test_split  # Scikit-learn's function to split data into train and test sets
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification  # HuggingFace Transformers for tokenization and model
from transformers import Trainer, TrainingArguments  # HuggingFace Trainer API for model training and fine-tuning
import torch  # PyTorch for tensor operations and model handling
from sklearn.metrics import classification_report, confusion_matrix  # Scikit-learn for generating classification metrics

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("himanshunayal/intent-recognition-dataset", path="train.csv")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/intent-recognition-dataset/train.csv


In [4]:
import pandas as pd

# Update the file path to point to the specific dataset file
file_path = "/kaggle/input/intent-recognition-dataset/train.csv"

# Load the CSV file into a DataFrame
train = pd.read_csv(file_path)

# Preview the first few rows of the dataset
print(train.head())


                                                text         intent
0   listen to westbam alumb allergic on google music      PlayMusic
1         add step to me to the 50 clásicos playlist  AddToPlaylist
2  i give this current textbook a rating value of...       RateBook
3               play the song little robin redbreast      PlayMusic
4  please add iris dement to my playlist this is ...  AddToPlaylist


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13084 entries, 0 to 13083
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    13084 non-null  object
 1   intent  13084 non-null  object
dtypes: object(2)
memory usage: 204.6+ KB


In [8]:
train['intent'].value_counts()

,count
intent,
PlayMusic,1914
GetWeather,1896
BookRestaurant,1881
RateBook,1876
SearchScreeningEvent,1852
SearchCreativeWork,1847
AddToPlaylist,1818


**TEXT PREPARATION**

In [9]:
train['intent'] = train['intent'].apply(lambda x: x.lower())

In [10]:
import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords (only need to do this once)
nltk.download('stopwords')
# Load the list of stopwords
stop_words = set(stopwords.words('english'))


# Preprocessing function: convert text to lowercase and remove stopwords
def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove stopwords: split the text, filter out stopwords, and join back
    text = ' '.join([word for word in text.split() if word not in stop_words])

    return text

# Example usage
text_input = 'Stop to play music'

# Preprocess the text
processed_text = preprocess_text(text_input)
print(f"Processed Text: {processed_text}")


Processed Text: stop play music


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [11]:
train['text'] = train['text'].apply(preprocess_text)

**Creating Label**

In [12]:
# Check the unique labels in the 'Intent_Label' column
unique_labels = train['intent'].unique()
print("unique labels :", pd.unique)

# Create a mapping from labels to numbers (numeric encoding)
label_to_id = {label: i for i, label in enumerate(unique_labels)}

# Map the 'Intent_Label' to numeric labels in the 'Label' column
train['Label'] = train['intent'].map(label_to_id)

# Check the updated DataFrame
train.head()


unique labels : <function unique at 0x7f0f0b71eca0>


,text,intent,Label
0,listen westbam alumb allergic google music,playmusic,0
1,add step 50 clásicos playlist,addtoplaylist,1
2,give current textbook rating value 1 best rati...,ratebook,2
3,play song little robin redbreast,playmusic,0
4,please add iris dement playlist selena,addtoplaylist,1


In [15]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [16]:
from datasets import Dataset

**TOKENIZATION**

In [17]:
# Load the DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")  # Load pre-trained DistilBERT tokenizer

# Calculate the maximum tokenized length from the dataset
max_length = max([len(tokenizer.encode(text)) for text in train['text']])  # Tokenize and count the length of each tokenized text
print("Max Length:", max_length)

# Tokenization function with labels
def tokenize_function(examples):  # Define a function to tokenize inputs and add labels
    tokenized_input = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=max_length)  # Use the max_length calculated above
    tokenized_input['labels'] = examples['Label']  # Add labels to the tokenized data for supervised training
    return tokenized_input  # Return tokenized data with labels


dataset = Dataset.from_pandas(train[['text', 'Label']])  # Convert the DataFrame into a HuggingFace Dataset

# Apply tokenization
dataset = dataset.map(tokenize_function, batched=True)  # Apply the tokenization function to the dataset

# Check tokenized data
dataset[0]  # Display the tokenized version of the first example in the dataset to verify the transformation


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Max Length: 28


Map:   0%|          | 0/13084 [00:00<?, ? examples/s]

{'text': 'listen westbam alumb allergic google music',
 'Label': 0,
 'input_ids': [101,
  4952,
  2225,
  3676,
  2213,
  2632,
  25438,
  27395,
  8224,
  2189,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'labels': 0}

**Finetuning DistilBERT Model**

In [18]:
# Initialize DistilBERT model for sequence classification
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=len(unique_labels))

# Move model to GPU if available
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [20]:
from transformers import TrainingArguments, Trainer  # Import necessary classes from Hugging Face

# Training arguments
training_args = TrainingArguments(  # Define the training configurations
    output_dir="./results",  # Directory to save results (model checkpoints, logs, etc.)
    eval_strategy="epoch",  # Evaluate the model at the end of each epoch
    learning_rate=2e-5,  # Set learning rate for the optimizer
    per_device_train_batch_size=16,  # Batch size for training (number of examples per device)
    per_device_eval_batch_size=64,  # Batch size for evaluation
    num_train_epochs=5,  # Number of epochs to train the model
    weight_decay=0.01,  # L2 regularization to avoid overfitting
    logging_dir="./logs",  # Directory to store training logs
    logging_steps=10,  # Log training information every 10 steps
)

# Trainer setup
trainer = Trainer(  # Initialize the Trainer with the model and training configurations
    model=model,  # Model to be trained
    args=training_args,  # Training arguments
    train_dataset=dataset,  # Training dataset
    eval_dataset=dataset,  # Validation dataset (in practice, this should be a separate dataset)
)

# Train the model
trainer.train()  # Start training the model based on the provided training arguments


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: muadh (muadh-kiit-deemed-to-be-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.035600,0.034883
2,0.002100,0.021522
3,0.052500,0.012592
4,0.002300,0.006112
5,0.000500,0.003617


TrainOutput(global_step=4090, training_loss=0.06324597640363795, metrics={'train_runtime': 461.3489, 'train_samples_per_second': 141.802, 'train_steps_per_second': 8.865, 'total_flos': 473965075426080.0, 'train_loss': 0.06324597640363795, 'epoch': 5.0})

In [21]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate the model on the test dataset
predictions, true_labels, _ = trainer.predict(dataset)

# Convert predictions to label indices
predicted_labels = predictions.argmax(axis=1)

# Generate classification report and confusion matrix
print("Classification Report:")
print(classification_report(true_labels, predicted_labels))

print("Confusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1914
           1       1.00      1.00      1.00      1818
           2       1.00      1.00      1.00      1876
           3       1.00      1.00      1.00      1852
           4       1.00      1.00      1.00      1881
           5       1.00      1.00      1.00      1896
           6       1.00      1.00      1.00      1847

    accuracy                           1.00     13084
   macro avg       1.00      1.00      1.00     13084
weighted avg       1.00      1.00      1.00     13084

Confusion Matrix:
[[1914    0    0    0    0    0    0]
 [   2 1816    0    0    0    0    0]
 [   0    0 1876    0    0    0    0]
 [   0    0    0 1850    0    0    2]
 [   0    0    0    0 1881    0    0]
 [   0    0    0    0    4 1892    0]
 [   1    0    0    4    0    0 1842]]


In [22]:
# Save the model and tokenizer
model.save_pretrained('./saved_fine_tuned_model')
tokenizer.save_pretrained('./saved_fine_tuned_model')

('./saved_fine_tuned_model/tokenizer_config.json',
 './saved_fine_tuned_model/special_tokens_map.json',
 './saved_fine_tuned_model/vocab.txt',
 './saved_fine_tuned_model/added_tokens.json')

In [23]:
# Load the fine-tuned model and tokenizer for inference
model = DistilBertForSequenceClassification.from_pretrained('./saved_fine_tuned_model')
tokenizer = DistilBertTokenizer.from_pretrained('./saved_fine_tuned_model')

In [24]:
# Now create the reverse mapping for inference
id_to_label = {i: label for label, i in label_to_id.items()}  # Reverse the mapping
id_to_label

{0: 'playmusic',
 1: 'addtoplaylist',
 2: 'ratebook',
 3: 'searchscreeningevent',
 4: 'bookrestaurant',
 5: 'getweather',
 6: 'searchcreativework'}

In [25]:
# Function to make a prediction
def predict(text, model, tokenizer, max_length=21):
    # Preprocess the input text
    text = preprocess_text(text)
    # Tokenize the input text
    inputs = tokenizer(text, padding='max_length', truncation=True, max_length=max_length, return_tensors="pt")

    # Make prediction
    with torch.no_grad():  # Disable gradient calculation for inference
        outputs = model(**inputs)  # Get model output
        logits = outputs.logits  # Get logits from the output

    # Get the predicted label (highest logit)
    predicted_class_id = torch.argmax(logits, dim=-1).item()  # Get the index of the max logit
    return predicted_class_id

In [27]:
# Define the intent labels
id_to_label = {
    0: 'getweather',
    1: 'searchcreativework',
    2: 'searchscreeningevent',
    3: 'addtoplaylist',
    4: 'bookrestaurant',
    5: 'ratebook',
    6: 'playmusic'
}

# Example messages for testing
test_messages = [
    "What's the weather like today?",
    "Find me a creative project about AI.",
    "Are there any events screening this weekend?",
    "Add this song to my playlist.",
    "I'd like to book a table for two.",
    "Rate the book I just finished reading.",
    "Play some relaxing music.",
    "Can you find a documentary on climate change?",
    "What time does the movie start tonight?",
    "Add the new album to my library."
]

# Test the model with the example messages
for message in test_messages:
    predicted_label = predict(message, model, tokenizer)
    predicted_intent = id_to_label.get(predicted_label, "Unknown Intent")
    print(f"Message: {message}")
    print(f"Predicted Label: {predicted_label}, Predicted Intent: {predicted_intent}\n")


Message: What's the weather like today?
Predicted Label: 5, Predicted Intent: ratebook

Message: Find me a creative project about AI.
Predicted Label: 6, Predicted Intent: playmusic

Message: Are there any events screening this weekend?
Predicted Label: 3, Predicted Intent: addtoplaylist

Message: Add this song to my playlist.
Predicted Label: 1, Predicted Intent: searchcreativework

Message: I'd like to book a table for two.
Predicted Label: 4, Predicted Intent: bookrestaurant

Message: Rate the book I just finished reading.
Predicted Label: 2, Predicted Intent: searchscreeningevent

Message: Play some relaxing music.
Predicted Label: 0, Predicted Intent: getweather

Message: Can you find a documentary on climate change?
Predicted Label: 6, Predicted Intent: playmusic

Message: What time does the movie start tonight?
Predicted Label: 3, Predicted Intent: addtoplaylist

Message: Add the new album to my library.
Predicted Label: 1, Predicted Intent: searchcreativework

